# 🌳 Modelo Global XGBoost — Região Sul (Todas as Latitudes)

Este notebook implementa o **Plano Mestre XGBoost**:
1. Engole os 1.190 municípios de uma vez só.
2. Cria as colunas biológicas e geográficas (`max_historico` como proxy de população, `casos_lag1` para contágio, e `casos_lag12` para imunidade de rebanho).
3. Treina o modelo em 2015–2023.
4. Realiza o teste de fogo prestando o ano de **2024** (o maior surto da história) cegas.

## 1. Setup e Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import warnings
from pathlib import Path
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 120

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

print("Ambiente XGBoost configurado!")

Ambiente XGBoost configurado!


## 2. Carregamento do Painel de Dados (142.000 linhas)

In [2]:
from src.processamento.motor_lote_modelagem_v2_por_latitude import build_base_panel

print("Carregando dados de todos os municípios...")
painel, lookup = build_base_panel(BASE_DIR)

print(f"Total de Registros Iniciais: {len(painel):,}")
print(f"Total de Municípios: {painel['code'].nunique()}")
painel.head(3)

17:45:06 [INFO] build_base_panel → fonte canônica: output/csvs_latitude/ (1191 municípios esperados)


Carregando dados de todos os municípios...


Total de Registros Iniciais: 142,800
Total de Municípios: 1190


,code,year,month,dengue_cases,latitude,city_name,lat_faixa_id,temp_media,precipitacao,enso
0,410010,2015,1,0.0,-23.307179,Abatiá,2,25.2,164.2,0.5
1,410010,2015,2,1.0,-23.307179,Abatiá,2,23.2,112.3,0.5
2,410010,2015,3,8.0,-23.307179,Abatiá,2,22.2,128.2,0.5


## 3. Engenharia de Features (A Ciência do Especialista)

Aqui nós construímos a "inteligência" do modelo sem precisar baixar nada novo, usando as matemática da própria série.

In [3]:
df = painel.copy()

# 1. Criar coluna de Data e ordenar rigorosamente por cidade e tempo
df['data'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2))
df = df.sort_values(['code', 'data']).reset_index(drop=True)

# 2. Proxy de População / Urbanização (Máximo Histórico da Cidade)
# Isso ensina a árvore a separar a "Metrópole" da "Vila"
max_hist = df.groupby('code')['dengue_cases'].max().rename('max_historico')
df = df.merge(max_hist, on='code')

# 3. Lags Climáticos (3 meses = Tempo de chuva até surto)
df['temp_lag3'] = df.groupby('code')['temp_media'].shift(3)
df['prec_lag3'] = df.groupby('code')['precipitacao'].shift(3)
df['enso_lag3'] = df.groupby('code')['enso'].shift(3)

# 4. Biologia Epidemiológica (Autoregressão)
df['casos_lag1'] = df.groupby('code')['dengue_cases'].shift(1)  # Velocidade de Contágio Atual
df['casos_lag12'] = df.groupby('code')['dengue_cases'].shift(12) # Imunidade de Rebanho / Estacionalidade pura

# Remover linhas vazias (os primeiros 12 meses de todas as cidades serão perdidos devido ao lag12)
df_ml = df.dropna().copy()

print(f"Dados prontos para ML: {len(df_ml):,} linhas (após lag)")

Dados prontos para ML: 128,520 linhas (após lag)


## 4. Divisão: Treino vs Teste Cego (2024)

In [4]:
# Features de entrada da Árvore (7 inputs incríveis)
FEATURES = [
    'temp_lag3', 'prec_lag3', 'enso_lag3',   # Clima
    'latitude',                              # Geografia / Barreira da Serra
    'max_historico',                         # Proxy de Densidade Populacional
    'casos_lag1', 'casos_lag12'              # Epidemiologia
]
TARGET = 'dengue_cases'

# Treino: Tudo antes de 2024
treino = df_ml[df_ml['year'] < 2024].copy()
# Teste Cego: O ano da grande epidemia (2024)
teste = df_ml[df_ml['year'] == 2024].copy()

X_treino = treino[FEATURES]
y_treino = treino[TARGET]
X_teste = teste[FEATURES]
y_teste = teste[TARGET]

print(f"Linhas de Treino: {len(X_treino):,}")
print(f"Linhas de Teste Cego (2024): {len(X_teste):,}")

Linhas de Treino: 114,240
Linhas de Teste Cego (2024): 14,280


## 5. O Treinamento do Gigante (XGBoost)

In [5]:
# Configurar a IA
modelo = xgb.XGBRegressor(
    n_estimators=300,        # Número de árvores
    learning_rate=0.05,      # Passo de aprendizado
    max_depth=6,             # Profundidade (inteligência) das árvores
    subsample=0.8,           # Amostragem para evitar vício
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

print("Treinando XGBoost Global... (isso pode levar alguns segundos)")
modelo.fit(X_treino, y_treino)
print("\u2705 Treinamento Concluído!")

Treinando XGBoost Global... (isso pode levar alguns segundos)


✅ Treinamento Concluído!


### 5.1 Importância das Variáveis (O que a Árvore mais usou?)

In [6]:
importancias = pd.Series(modelo.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(10, 5))
importancias.plot(kind='barh', color='darkgreen', edgecolor='black')
plt.title('Raio-X da Inteligência do XGBoost (Feature Importance)')
plt.xlabel('Peso de Importância (%)')
plt.tight_layout()
plt.savefig(BASE_DIR / 'output' / 'xgboost_importancia_variaveis.png', bbox_inches='tight')
plt.show()

## 6. Previsão e Métricas Cegas (2024)

In [7]:
predicoes = modelo.predict(X_teste)
# Limpar numéros negativos (não existe caso negativo)
predicoes = np.clip(predicoes, 0, None)

teste['previsto_xgb'] = predicoes

mae = mean_absolute_error(y_teste, predicoes)
r2 = r2_score(y_teste, predicoes)

print("== RESULTADOS DO TESTE CEGO (ANO 2024 NO SUL INTEIRO) ==")
print(f"R\u00b2 Score Global: {r2:.3f} (Quanta variação ele acertou? >0.5 é excelente!)")
print(f"MAE (Erro Absoluto Médio): {mae:.1f} casos / cidade / mês")

== RESULTADOS DO TESTE CEGO (ANO 2024 NO SUL INTEIRO) ==
R² Score Global: 0.327 (Quanta variação ele acertou? >0.5 é excelente!)
MAE (Erro Absoluto Médio): 64.9 casos / cidade / mês


### 6.1 Visão Agregada Regional (A Região Sul Acumulada em 2024)

In [8]:
# Soma todos os municipios mes a mes no ano de teste
teste_agg = teste.groupby('month')[['dengue_cases', 'previsto_xgb']].sum().reset_index()

plt.figure(figsize=(12, 6))
plt.plot(teste_agg['month'], teste_agg['dengue_cases'], 'k-o', linewidth=3, markersize=8, label='Dengue REAL (Sul Acumulado)')
plt.plot(teste_agg['month'], teste_agg['previsto_xgb'], 'g--s', linewidth=3, markersize=8, label='Previs\u00e3o XGBoost Global')

plt.title('Teste Cego (2024): Modelo Global vs Casos Reais na Regi\u00e3o Sul', fontsize=14, fontweight='bold')
plt.xlabel('M\u00eas (2024)')
plt.ylabel('Total de Casos na Regi\u00e3o Sul')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig(BASE_DIR / 'output' / 'xgboost_teste_cego_regional_2024.png', bbox_inches='tight')
plt.show()

### 6.2 Visão Local (Top 3 Cidades com Surtos)

In [9]:
# Pegar as 3 maiores do ano de 2024
top3_2024 = teste.groupby(['code', 'city_name'])['dengue_cases'].sum().reset_index().sort_values('dengue_cases', ascending=False).head(3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, row in enumerate(top3_2024.itertuples()):
    df_cidade = teste[teste['code'] == row.code].sort_values('month')
    
    axes[i].plot(df_cidade['month'], df_cidade['dengue_cases'], 'k-o', linewidth=2, label='Real')
    axes[i].plot(df_cidade['month'], df_cidade['previsto_xgb'], 'g--x', linewidth=2, label='Previsto (XGBoost)')
    axes[i].set_title(f"{row.city_name.upper()}\n(Surto 2024)", fontweight='bold')
    axes[i].set_xticks(range(1, 13))
    axes[i].legend()

plt.tight_layout()
plt.savefig(BASE_DIR / 'output' / 'xgboost_top3_cidades_2024.png', bbox_inches='tight')
plt.show()